<h1 align="center"> pyeCE tutorial </h1>
<h3 align="center">Yann Lorris Mueller and Anirudh Raju Natarajan</h3>


<center><img src="Figures/pyece_logo.png" height="200px"></center>

In [1]:
# Load parser libraries
import yaml
import json
import pandas as pd

# Load libraries to perform mathematical tasks
import numpy as np
from scipy.spatial import ConvexHull, convex_hull_plot_2d
from scipy.integrate import cumulative_simpson

# Load pymatgen functionalities to deal with crystal structures
from pymatgen.core import Structure

# Load pyece functionalities
from pyece.core.clex import eCE
from pyece.learn.data import ConfigData, DataBase, ClexDataset

# Load pyplot
import matplotlib.pyplot as plt
    

## 1) Construct an eCE model

### a) Set up the primitive structure

The **embedded cluster expansion** (eCE) is an **on-lattice** atomistic model. Atoms are assumed to sit on lattice sites and the model only captures the configurational degree of freedom.
As a first step, one must specify the crystal structure and the species permitted at each lattice site.

In **pyeCE**, the system is defined by a PRIM file, similar to a VASP POSCAR file, with the permitted species indicated for each atomic site.
To create the PRIM file, one needs to specify:
* **The lattice**: 3 linearly independent vectors that define the primitive cell
* **The basis sites**: N 3-D vectors indicating the positions of the atoms within the primitive cell
* **The degrees of freedom**: permitted species for each atomic site

<img src="Figures/prim.png" height="300px">

<div style="background:#e8f4f8; border-left:4px solid #2196F3; padding:10px; margin:8px 0">
<strong>Example — FCC ternary system</strong><br><br>
In this example, we construct the PRIM object that defines the primitive cell and the chemical degrees of freedom. The system consists of an FCC lattice where lattice sites can be occupied by elements of group 11 (Cu, Ag, and Au).<br><br>
We use the <code>Structure</code> class in Pymatgen to perform this task.
</div>

In [ ]:
# Parameters
species = ['Ag', 'Au', 'Cu']
a = 3.5727932
fcc_lattice = [
    [a/2, a/2,   0],
    [  0, a/2, a/2],
    [a/2, 0,   a/2]
]
coords = [[0.0, 0.0, 0.0]]

# Create POSCAR file
elem = 'Ag'
primitive = Structure(
    lattice = fcc_lattice,
    species = [elem]*len(coords), 
    coords  = coords
)
prim_file = primitive.to(fmt='poscar').split('\n')

# Create pyece PRIM file
prim_file[0] = ' '.join(species)
prim_file.pop(5)
for index in range(7, len(prim_file)):
    prim_file[index] = prim_file[index].replace(elem, ' '.join(species))

# Save pyece PRIM file
with open('PRIM_FCC.vasp', 'w') as f:
    f.write('\n'.join(prim_file))
print(*prim_file,sep='\n')

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:10px; margin:8px 0">
<strong>Task — BCC senary system</strong><br><br>
Create the PRIM file for a BCC primitive structure where atomic sites can be occupied by the 6 species of groups 5 and 6, namely Cr, Mo, Nb, Ta, V, and W, with a lattice parameter $a=3.5727932$.
</div>

In [ ]:
# Write your solution here

<details>
<summary><strong>▶ Solution</strong></summary>

```python
# Parameters
species = ['Cr', 'Mo', 'Nb', 'Ta', 'V', 'W']
a = 3.5727932
bcc_lattice = [
    [-a/2, -a/2,  a/2],
    [-a/2,  a/2, -a/2],
    [ a/2, -a/2, -a/2]
]
coords = [[0.0, 0.0, 0.0]]

# Create POSCAR file
elem = 'Cr'
primitive = Structure(
    lattice = bcc_lattice,
    species = [elem]*len(coords), 
    coords  = coords
)
prim_file = primitive.to(fmt='poscar').split('\n')

# Create pyece PRIM file
prim_file[0] = ' '.join(species)
prim_file.pop(5)
for index in range(7, len(prim_file)):
    prim_file[index] = prim_file[index].replace(elem, ' '.join(species))

# Save pyece PRIM file
with open('PRIM_BCC.vasp', 'w') as f:
    f.write('\n'.join(prim_file))
print(*prim_file,sep='\n')
```

</details>

### b) Set up the settings to build the eCE model

Once the system is defined, one needs to determine how atoms interact with each other, i.e., the Hamiltonian of the system. Cluster expansions act as highly efficient surrogate models that evaluate the energy of on-lattice atomic arrangements. **pyeCE** models require specifying:
* **Types of interactions**: one specifies the types of interaction considered. One typically includes all pair interactions within a sphere of radius $r_\text{pair}$, all triplet interactions within a sphere of radius $r_\text{triplet}$, etc.
* **Site-basis functions**: one also needs to define the mapping between atomic species and vectors spanning the $k$-dimensional space $\mathbb{R}^k$. This requires specifying the type of basis vectors (e.g., Chebyshev or occupational) and the number of dimensions $k$. In traditional cluster expansion, $k$ equals the number of elements. Within the embedded cluster expansion framework, $k$ is smaller and species are projected into a lower-dimensional space.
* **Neural network**: specific to the embedded cluster expansion framework, the descriptors serve as input to a neural network that evaluates the energy of the structure. The network is typically a simple feedforward network.

<img src="Figures/build.png" height="400px">

<div style="background:#e8f4f8; border-left:4px solid #2196F3; padding:10px; margin:8px 0">
<strong>Example — FCC ternary eCE model</strong><br><br>
In this example, we create an eCE model with pair interactions up to $r_\text{pair} = 6.0\,\text{Å}$, triplet interactions up to $r_\text{triplet} = 4.5\,\text{Å}$, and quadruplet interactions up to $r_\text{quadruplet} = 4.0\,\text{Å}$, using a Chebyshev basis, for the FCC ternary system described above. The number of embedding dimensions is set to 2 ($k=2$). The feedforward neural network has 2 layers with 128 and 64 nodes, respectively, and ReLU activation functions between layers.<br><br>
We use the CLI command <code>ece build</code> to perform this task.
</div>

In [ ]:
# Parameters
path_to_PRIM = 'PRIM_FCC.vasp'
radii = [0.0, 0.0, 6.0, 4.5, 4.0]
basis = 'chebyshev'
k = 2
nn_layers = [128, 64]
activation = 'ReLU'
model_name = 'model_FCC.pth'

# Create settings file
orbit_tree_specs = {
    "orbit_tree_length_filter": {str(body_order): {'max_length': radius} for body_order, radius in enumerate(radii)}
}
settings = {
    # Prim specifications
    "prim" : path_to_PRIM,
    "basis": basis,
    "orbit_tree_specs": orbit_tree_specs,

    # Neural network specifications
    "embedding_dimensions": [k],
    "nn_layers": nn_layers,
    "activation": activation,
    
    # Model
    "name": model_name,
}
with open('build_settings.yaml', 'w') as f:
    yaml.dump(settings, f, default_flow_style=False)
    
# Build the eCE model using the built-in CLI function
! ece build -s build_settings.yaml


You can print information about the `Prim`, `SiteBasis`, and `Features` of the eCE model using the CLI command `ece print` with the `-p` flag for the Prim, the `-s` flag for the SiteBasis, and the `-f` flag for the Features.

In [ ]:
! ece print --model model_FCC.pth -p -s -f

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:10px; margin:8px 0">
<strong>Task — BCC senary eCE model</strong><br><br>
Create an eCE model for the BCC senary system (<code>PRIM_BCC.vasp</code>) using the following settings:
<ul>
<li><strong>Basis</strong>: chebyshev</li>
<li><strong>Pair interactions</strong>: up to 8 Å</li>
<li><strong>Triplet interactions</strong>: up to 3.6 Å</li>
<li><strong>Embedding dimensions</strong>: $k =$ 3</li>
<li><strong>Neural network layers</strong>: 128 x 128 x 8</li>
<li><strong>Activation function</strong>: ReLU</li>
</ul>
Then use <code>ece print</code> to inspect the Prim, SiteBasis, and Features of the model.
</div>

In [ ]:
# Write your solution here

<details>
<summary><strong>▶ Solution</strong></summary>

```python
path_to_PRIM = 'PRIM_BCC.vasp'
radii      = [0.0, 0.0, 8.0, 3.6]
basis      = 'chebyshev'
k          = 3
nn_layers  = [128, 128, 8]
activation = 'ReLU'
model_name = 'model_BCC.pth'

# Create settings file
orbit_tree_specs = {
    "orbit_tree_length_filter": {str(body_order): {'max_length': radius} for body_order, radius in enumerate(radii)}
}
settings = {
    "prim": path_to_PRIM,
    "basis": basis,
    "orbit_tree_specs": orbit_tree_specs,
    "embedding_dimensions": [k],
    "nn_layers": nn_layers,
    "activation": activation,
    "name": model_name,
}
with open('build_settings.yaml', 'w') as f:
    yaml.dump(settings, f, default_flow_style=False)

# Build the eCE model
! ece build -s build_settings.yaml

# Inspect the model
! ece print --model model_BCC.pth -p -s -f
```

</details>

## 2) Train the eCE model

### a) Construct a training database

eCE models must be trained on data. These data typically consist of symmetrically unique atomic orderings in small supercells of the primitive structure.

<img src="Figures/data.png" height="600px">

A database of symmetrically unique orderings can be constructed with an enumeration algorithm implemented in Pymatgen's `EnumerateStructureTransformation`. Energies are then computed with DFT. 

We will use the formation energies of over 3600 symmetrically unique arrangements of group 5 and 6 elements on the BCC lattice, stored in `Files/database.pkl`. Each entry includes:
* **Composition**: columns named `x_ELEM` for each element
* **Property**: the `property` column holds the formation energy per atom
* **Ordering**: the `ConfigData` column contains the input object that describes the specific atomic arrangement

Before fitting, we typically analyse the system by plotting binary **convex hulls** — phase diagrams at 0 K that show ground states and common tangent constructions.

The command below prints the contents of the database.

In [ ]:
# Note: if model_BCC.pth did not build correctly on your machine,
# you can use Files/model_BCC.pth (the pre-trained model) for all remaining exercises.
! ece print --model Files/model_BCC.pth -d Files/database.pkl

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:10px; margin:8px 0">
<strong>Task — Binary convex hull</strong><br><br>
Plot the convex hull of the Mo–Ta binary system. You can use the <code>ConvexHull</code> and <code>convex_hull_plot_2d</code> functions from SciPy to compute and plot the convex hull, respectively.
</div>

In [ ]:
# Write your solution here


<details>
<summary><strong>▶ Solution</strong></summary>

```python
db = DataBase.from_pickle('Files/database.pkl')
elements = ['Mo', 'Ta']

# Select the binary data
db = db[(db[['x_%s'%elem for elem in elements]].sum(axis=1)-1.0).abs()< 1e-6]

# Construct convex hull
data = db[['x_%s'%elements[1], 'property']].to_numpy()
hull = ConvexHull(data)

fig, ax = plt.subplots()
convex_hull_plot_2d(hull=hull, ax=ax)
plt.axis([0, 1, None, None])
plt.annotate(elements[0], xy=(0, -0.1), xycoords='axes fraction', ha='center', va='top', fontsize=15, fontweight='bold')
plt.annotate(elements[1], xy=(1, -0.1), xycoords='axes fraction', ha='center', va='top', fontsize=15, fontweight='bold')
plt.xlabel('x')
plt.ylabel('Formation energy (eV/atom)')
plt.tight_layout()
plt.show()
```

</details>

### b) Fit the eCE model

With a model and a training database in hand, we can train the model using gradient descent as implemented in PyTorch. **Ladder training** is used to improve convergence toward the global minimum. It performs several successive training runs, starting with a small set of descriptors (e.g., first- and second-nearest neighbour pair interactions) and incrementally adding more until all descriptors are included.

The figure below illustrates a 3-step ladder training: first- and second-nearest neighbour pairs → third- and fourth-nearest neighbour pairs → triplets.

<img src="Figures/ladder_training_illustration.png" height="300px">

The key training settings are:
* **train_prop**: fraction of data used for training
* **valid_prop**: fraction of data used for validation
* **test_prop**: fraction of data used for testing
* **batch_size**: number of data points per batch
* **loss_fct**: loss function (typically `MSELoss` from PyTorch)
* **ladder_steps**: list indicating the number of orbits included at each ladder step (see `ece print --model model_BCC.pth -f`)
* **optimizers**: optimizer name and options (typically `Adam`)
* **epochs**: number of epochs per ladder step
* **lr**: learning rate per ladder step
* **schedulers**: learning-rate scheduler and options (typically `MultiStepLR`)
* **earlystopping_patience**: number of epochs without improvement before stopping, per ladder step

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:10px; margin:8px 0">
<strong>Task — Fit the BCC senary eCE model</strong><br><br>
Train the eCE model for the senary BCC system. The cell below implements the following (relatively coarse) settings:
<ul>
<li><strong>train_prop</strong>: 60%</li>
<li><strong>valid_prop</strong>: 10%</li>
<li><strong>test_prop</strong>: 30%</li>
<li><strong>batch_size</strong>: 50</li>
<li><strong>loss_fct</strong>: MSELoss</li>
<li><strong>ladder_steps</strong>: [11] — single step with all orbits</li>
<li><strong>optimizers</strong>: Adam with amsgrad</li>
<li><strong>epochs</strong>: [100]</li>
<li><strong>lr</strong>: [1e-3]</li>
<li><strong>schedulers</strong>: MultiStepLR, gamma=0.1, milestones=[50, 80]</li>
<li><strong>earlystopping_patience</strong>: [10]</li>
</ul>
Modify these settings to see if you can achieve a lower validation error. We use the CLI command <code>ece fit</code> to perform this task.
</div>

In [ ]:
# Parameters
database_name = 'Files/database.pkl'
model_name = 'Files/model_BCC.pth'
training_proportion = 0.6
validation_proportion = 0.1
testing_proportion = 0.3
batch_size = 50
loss_fct = 'MSELoss'
ladder_steps = [11]
optimizer_name = 'Adam'
list_optimizers = [{optimizer_name: {'amsgrad': True}} for step in range(len(ladder_steps))]
scheduler_name = 'MultiStepLR'
list_schedulers = [{scheduler_name: {'milestones': [50, 80], 'gamma': 0.1}}]
epochs = [100]
lr = [1e-3]
earlystopping_patience = [10]
device = 'cpu'

# Training settings
settings = {
    # Database specifications
    'data_file': database_name,
    'train_prop': training_proportion,
    'valid_prop': validation_proportion,
    'test_prop': [testing_proportion],

    # Gradient descent specification
    'ladder_steps': ladder_steps,
    'epochs': epochs,
    'lr': lr,
    'optimizers': list_optimizers,
    'schedulers': list_schedulers,
    'earlystopping_patience': earlystopping_patience,
    'loss_fct': loss_fct,
    'batch_size': batch_size,
    
    # Model
    'model': model_name,
    "name": model_name,
    'device': device,
    
    # Verbosity
    'verbose': True
}
with open('train_settings.yaml', 'w') as f:
    yaml.dump(settings, f, default_flow_style=False)

# Fit the eCE model using the built-in CLI function
! ece fit -s train_settings.yaml

<details>
<summary><strong>▶ Solution</strong></summary>

```python
# Parameters
database_name = 'Files/database.pkl'
model_name = 'Files/model_BCC.pth'
training_proportion = 0.6
validation_proportion = 0.1
testing_proportion = 0.3
batch_size = 50
loss_fct = 'MSELoss'
ladder_steps = [4, 6, 8, 10, 11]
optimizer_name = 'Adam'
list_optimizers = [{optimizer_name: {'amsgrad': True}} for step in range(len(ladder_steps))]
scheduler_name = 'MultiStepLR'
list_schedulers = [{scheduler_name: {'milestones': [15, 35] if step > 0 else [55, 85, 95], 'gamma': 0.1}} for step in range(len(ladder_steps))]
epochs = [100, 40, 40, 40, 40]
lr = [1e-3 if step > 0 else 1e-2 for step in range(len(ladder_steps))]
earlystopping_patience = [100, 40, 40, 40, 40]
device = 'cpu'

settings = {
    'data_file': database_name,
    'train_prop': training_proportion,
    'valid_prop': validation_proportion,
    'test_prop': [testing_proportion],
    'ladder_steps': ladder_steps,
    'epochs': epochs,
    'lr': lr,
    'optimizers': list_optimizers,
    'schedulers': list_schedulers,
    'earlystopping_patience': earlystopping_patience,
    'loss_fct': loss_fct,
    'batch_size': batch_size,
    'model': model_name,
    "name": model_name,
    'device': device,
    'verbose': True
}
with open('train_settings.yaml', 'w') as f:
    yaml.dump(settings, f, default_flow_style=False)

! ece fit -s train_settings.yaml
```

</details>

### c) Compare convex hulls

Once trained, the model's accuracy should be assessed. A standard check in materials science is to compare DFT and model convex hulls: correct reproduction of ground states indicates a reliable model.

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:10px; margin:8px 0">
<strong>Task — DFT vs. eCE convex hull</strong><br><br>
Plot the convex hull of the Mo–Ta binary system for both DFT and eCE formation energies. Use <code>ConvexHull</code> and <code>convex_hull_plot_2d</code> from SciPy, and the following pyeCE functions to evaluate energies:
<ul>
<li><code>eCE.load(path_to_model)</code> — load the model</li>
<li><code>ConfigData.from_dict(dict)</code> — load a configuration from its <code>ConfigData</code> column entry</li>
<li><code>ClexDataset.build_dataset(configdata)</code> — build model inputs</li>
<li><code>model(X.neighborhood_occupations).mean()</code> — evaluate the mean formation energy</li>
</ul>
</div>

In [ ]:
# Write your solution here


<details>
<summary><strong>▶ Solution</strong></summary>

```python
db = DataBase.from_pickle('Files/database.pkl')
model_name = 'Files/model_BCC.pth'
elements = ['Mo', 'Ta']
device = 'cpu'
fig, ax = plt.subplots()

# Select the binary data
db = db[(db[['x_%s'%elem for elem in elements]].sum(axis=1)-1.0).abs()< 1e-6]

# DFT
data = db[['x_%s'%elements[1], 'property']].to_numpy(copy=True)
hull = ConvexHull(data)
convex_hull_plot_2d(hull=hull, ax=ax)

# eCE
model = eCE.load(model_name)
configdata = [ConfigData.from_dict(dct) for dct in db['ConfigData']]
dataset = ClexDataset.build_dataset(configdata, device=device)
E = np.array([model(X.neighborhood_occupations).mean().numpy(force=True) for X, _, _ in dataset])
data[:, 1] = E
hull = ConvexHull(data)
convex_hull_plot_2d(hull=hull, ax=ax)

plt.axis([0, 1, None, None])
plt.annotate(elements[0], xy=(0, -0.1), xycoords='axes fraction', ha='center', va='top', fontsize=15, fontweight='bold')
plt.annotate(elements[1], xy=(1, -0.1), xycoords='axes fraction', ha='center', va='top', fontsize=15, fontweight='bold')
plt.xlabel('x')
plt.ylabel('Formation energy (eV/atom)')
plt.tight_layout()
plt.show()
```

</details>

## 3) Computing finite-temperature thermodynamic properties

### a) Canonical Monte Carlo simulation

Cluster expansions efficiently evaluate the energy $E(\sigma)$ of a configuration $\sigma$ and can be used to estimate the free energy:

$F = H - TS = -k_B T \ln Z$

via the canonical partition function:

$Z = \sum_\sigma \exp\!\left( -\frac{E_\sigma}{k_B T} \right)$

using Monte Carlo sampling.

pyeCE provides built-in Monte Carlo functionality. The key settings are:
* **ensemble**: the thermodynamic ensemble
* **increment**: the simulation trajectory — typically a temperature ramp or chemical potential scan — defined by an initial state, a final state, and the number of steps
* **composition**: the initial composition of the simulation cell
* **supercell**: the supercell transformation matrix applied to the primitive cell
* **properties**: properties to compute during the simulation (e.g., SRO) and their options
* **n_uncorrelated**: number of uncorrelated samples to collect

<div style="background:#e8f4f8; border-left:4px solid #2196F3; padding:10px; margin:8px 0">
<strong>Example — Mo–Ta equimolar at 1000 K</strong><br><br>
We perform a Canonical MC simulation at 1000 K for the equimolar Mo–Ta binary. The simulation cell uses the conventional BCC supercell (transformation [[0,1,1],[1,0,1],[1,1,0]]) repeated 5 times in each direction.<br><br>
We use the CLI command <code>ece mcmc</code> to perform this task.
</div>

In [ ]:
# Parameters
model_name = 'Files/model_BCC.pth'
ensemble = 'Canonical'
increment = [{'mode': 'linear', 'number': 1, 'initial': {'temperature': 1000}, 'final': {'temperature': 1000}}]
composition = [0.0, 0.5, 0.0, 0.5, 0.0, 0.0]
supercell = np.array([[0, 1, 1], [1, 0, 1], [1, 1, 0]]) * 5
properties = {'SRO': {'cutoffs': [0.0, 0.0, 3.5]}}
n_uncorrelated = 100
path_to_results = 'results_canonical_Mo-Ta.json'
device = 'cpu'

settings = {
    'model': model_name,
    'ensemble': ensemble,
    'increment': increment,
    'composition': composition,
    'supercell': supercell.tolist(),
    'properties': properties,
    'n_uncorrelated': n_uncorrelated,
    "path": path_to_results,
    'device': device,
    'verbose': True
}
with open('canonical_settings.yaml', 'w') as f:
    yaml.dump(settings, f, default_flow_style=False)

# Run the Canonical MC simulation
! ece mcmc -s canonical_settings.yaml


In [25]:
# Load the results
with open('Files/results_canonical_Mo-Ta.json') as f:
    data = json.load(f)
# Print results
H = data['<E>'][0]['mean']
print('<E> = %.3f eV/atom'%H)
sro = data['SRO_Mo-Ta_(Orbit_2_0)'][0]['mean']
print('Mo-Ta SRO = %.3f'%sro)

<E> = -0.163 eV/atom
Mo-Ta SRO = -0.433


<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:10px; margin:8px 0">
<strong>Task — Short-range order in the equiatomic senary alloy</strong><br><br>
Run a single Canonical Monte Carlo simulation for the equiatomic senary alloy stepping from 2000 K to 1000 K, then plot the first-nearest-neighbour SRO (cutoff radius $3.5\,\text{Å}$) at each temperature. Use the conventional BCC supercell (transformation [[0,1,1],[1,0,1],[1,1,0]]) repeated 5 times in each direction.
</div>

In [ ]:
# Write your solution here


<details>
<summary><strong>▶ Solution — Part 1: Run simulation</strong></summary>

```python
# Parameters
model_name = 'Files/model_BCC.pth'
ensemble = 'Canonical'
increment = [{'mode': 'linear', 'number': 2, 'initial': {'temperature': 2000}, 'final': {'temperature': 1000}}]
composition = [1/6, 1/6, 1/6, 1/6, 1/6, 1/6]
supercell = np.array([[0, 1, 1], [1, 0, 1], [1, 1, 0]]) * 5
properties = {'SRO': {'cutoffs': [0.0, 0.0, 3.5]}}
n_uncorrelated = 100
path_to_results = 'results_canonical.json'
device = 'cpu'

settings = {
    'model': model_name,
    'ensemble': ensemble,
    'increment': increment,
    'composition': composition,
    'supercell': supercell.tolist(),
    'properties': properties,
    'n_uncorrelated': n_uncorrelated,
    "path": path_to_results,
    'device': device,
    'verbose': True
}
with open('canonical_settings.yaml', 'w') as f:
    yaml.dump(settings, f, default_flow_style=False)

! ece mcmc -s canonical_settings.yaml
```

</details>

<details>
<summary><strong>▶ Solution — Part 2: Plot SRO</strong></summary>

```python
path_to_results = 'Files/results_canonical.json'
elements = ['Cr', 'Mo', 'Nb', 'Ta', 'V', 'W']
with open(path_to_results) as f:
    data = json.load(f)
list_pairs = []
list_sro = []
for i, elem1 in enumerate(elements):
    for elem2 in elements[i:]:
        sro = [dct['mean'] for dct in data['SRO_%s-%s_(Orbit_2_0)'%(elem1, elem2)]]
        list_pairs.append('-'.join([elem1, elem2]))
        list_sro.append(sro)
x = np.arange(len(list_pairs))
list_sro = np.array(list_sro).T
width = 0.25
fig, ax = plt.subplots(layout='constrained')
for i, temp in enumerate(data['temperature']):
    offset = width * i
    ax.bar(x + offset, list_sro[i], width, label='%.0fK'%temp)
ax.set_ylabel('SRO')
ax.set_xticks(x + width/2, list_pairs, rotation=90)
ax.legend(loc='lower right', ncols=1)
ax.axhline(0, color='k')
```

</details>

### b) Free energy integration

The finite-temperature free energy of a material can be evaluated by integrating the following equation:
$\frac{\partial (\beta F)}{\partial \beta} = \langle E \rangle$
where the free energy $F = \langle E \rangle - TS$, $\beta = 1/(k_B T)$ and $\langle\cdot\rangle$ denotes the ensemble average. Integrating with respect to $\beta$ gives:
$F(\beta, \vec{x}) = \frac{\beta_0}{\beta}F(\beta_0, \vec{x}) + \frac{1}{\beta} \int_{\beta_0}^{\beta} \langle E(b, \vec{x}) \rangle \, db$

Evaluating this integral requires the free energy at a reference temperature $\beta_{0}$. If $\beta_{0}$ is chosen to be a sufficiently high temperature, the free energy of the disordered phase can be approximated from its enthalpy and the ideal solution entropy. The ensemble averages of energy as a function of temperature are obtained from canonical Monte Carlo simulations where the simulation cell is cooled from a high temperature to a low temperature.

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:10px; margin:8px 0">
<strong>Task — Free energy from a canonical Monte Carlo simulation</strong><br><br>
Plot the evolution of the free energy $F$, enthalpy $H$, and entropy $S$ as a function of temperature. The energy per atom from a cooling simulation for the equiatomic senary alloy is provided in <code>Files/cooldown.dat</code>.<br><br>
Use <code>cumulative_simpson</code> for numerical integration and $k_B = 8.617333262\times10^{-5}\,\text{eV/K}$.
</div>

In [ ]:
# Write your solution here


<details>
<summary><strong>▶ Solution</strong></summary>

```python
# Parameters
filename = 'Files/cooldown.dat'
kB = 8.617333262e-5
x = np.array([1/6]*6)
Tmax = 3000

# Read results
data = pd.read_csv(filename, sep=' ', skipinitialspace=True)
T = data['temperature_(K)'].to_numpy()
H = data['energy_(eV/atom)'].to_numpy()
beta = 1/(kB*T)

# Initial values
S0 = -kB * (x * np.log(x, out=np.zeros_like(x), where=x > 0)).sum(axis=-1)
F0 = H[0] - T[0] * S0

# Integration
F = (cumulative_simpson(y=H, x=beta, initial=0) + F0 * beta[0]) / beta
S = (H - F) / T

# Plot
fig, ax = plt.subplots(3, 1, sharex=True)
mask = T <= Tmax
for i, (y, ylabel) in enumerate(zip([F, H, S], ['Free Energy\n(eV/atom)', 'Enthalpy\n(eV/atom)', 'Entropy\n(eV/K/atom)'])):
    plt.sca(ax[i])
    plt.plot(T[mask], y[mask])
    plt.ylabel(ylabel)
plt.xlabel('Temperature (K)')
plt.axis([0, Tmax, 0, S0])
plt.tight_layout()
plt.show()
```

</details>